# Milestone 1 — Profiling Dataset: Corpus-Indonesia

Notebook ini melakukan **profiling awal** dataset *Corpus-Indonesia* sesuai Milestone 1
Tugas 1 Analisis Big Data 2026.

Ketentuan yang dipenuhi notebook ini:

- Dataset: `data/raw/data/*.parquet` (7 file Parquet, ± 1,9 GB, 19.483.721 baris).
- Seluruh pembacaan data menggunakan **Polars Lazy API** (`pl.scan_parquet`), tanpa `read_parquet`
  ke memori penuh dan tanpa Pandas.
- Seluruh operasi bersifat agregasi/`head` kecil, sehingga memori yang terpakai tidak sebesar
  ukuran dataset.
- Data raw tidak diubah, dipindah, maupun dihapus.
- Jalankan notebook dari **root repository** (Docker: `docker run --rm -p 8888:8888 -v "$(pwd)":/home/jovyan/work tugas1-bigdata`).

Isi notebook:

1. Import library dan konfigurasi `DATA_PATH`
2. Pembacaan dataset dengan lazy evaluation
3. Schema/kolom dan tipe data
4. Jumlah baris dan jumlah kolom
5. Preview beberapa baris
6. Statistik dasar kolom teks
7. Pemeriksaan missing/null values
8. Jumlah data per file
9. Ringkasan hasil profiling

## 1. Import library dan konfigurasi path

In [1]:
import glob
import os
import time
from pathlib import Path

import polars as pl

print(f"polars: {pl.__version__}")

polars: 1.44.2


In [2]:
DATA_PATH = "data/raw/data/*.parquet"

# Fallback bila kernel Jupyter dimulai dari folder notebooks/ (cwd = root repository tidak berubah).
if not Path("data").exists() and Path("../data").exists():
    os.chdir("..")

files = sorted(glob.glob(DATA_PATH))

print(f"DATA_PATH      : {DATA_PATH}")
print(f"Working dir    : {Path.cwd()}")
print(f"File terdeteksi: {len(files)}")

if not files:
    raise FileNotFoundError(
        "File dataset tidak ditemukan. Jalankan notebook dari root repository "
        "dan pastikan data/raw/data/ berisi file Parquet."
    )

DATA_PATH      : data/raw/data/*.parquet
Working dir    : /home/jovyan/work
File terdeteksi: 7


## 2. Membaca dataset dengan lazy evaluation

`pl.scan_parquet()` tidak membaca data ke memori; ia hanya membangun rencana query (lazy plan).
Data baru diproses ketika query dieksekusi dengan `collect()`.

In [3]:
lf = pl.scan_parquet(DATA_PATH)

schema = lf.collect_schema()

print(schema)

Schema({'text': String})


## 3. Jumlah baris dan jumlah kolom

Jumlah baris dihitung dengan ekspresi lazy `pl.len()`, bukan dengan membaca seluruh dataset.

In [4]:
start = time.time()
jumlah_baris = lf.select(pl.len()).collect().item()
jumlah_kolom = len(schema.names())

print(f"Jumlah baris : {jumlah_baris:,}")
print(f"Jumlah kolom : {jumlah_kolom}")
print(f"Waktu query  : {time.time() - start:.2f} detik")

Jumlah baris : 19,483,721
Jumlah kolom : 1
Waktu query  : 0.03 detik


## 4. Preview beberapa baris

Hanya 5 baris pertama yang dikumpulkan ke memori, dan teks dipotong 120 karakter pertama
agar output tetap ringkas.

In [5]:
preview = (
    lf.head(5)
    .with_columns(pl.col("text").str.slice(0, 120).alias("text_120_karakter_pertama"))
    .select("text_120_karakter_pertama")
    .collect()
)

preview

text_120_karakter_pertama
str
"""Indonesia adalah negara kepula…"
"""Secara geografis, Indonesia be…"
"""Indonesia memiliki iklim tropi…"
"""Sejarah Indonesia sangat panja…"
"""Indonesia memproklamasikan kem…"


## 5. Tipe data setiap kolom

Schema diambil dari metadata Parquet (`collect_schema()`), tanpa membaca isi dataset.

In [6]:
tipe_data = pl.DataFrame(
    {
        "kolom": schema.names(),
        "tipe_data": [str(schema[name]) for name in schema.names()],
    }
)

tipe_data

kolom,tipe_data
str,str
"""text""","""String"""


## 6. Statistik dasar

Statistik relevan untuk kolom teks: panjang teks dalam karakter (min, median, mean, maks),
rata-rata jumlah kata, serta jumlah string kosong dan string yang hanya berisi spasi.
Seluruh nilai dihitung sebagai agregasi lazy sehingga hanya hasil agregat yang dikumpulkan.

In [7]:
start = time.time()
stats = lf.select(
    pl.len().alias("jumlah_baris"),
    pl.col("text").str.len_chars().min().alias("panjang_karakter_min"),
    pl.col("text").str.len_chars().median().alias("panjang_karakter_median"),
    pl.col("text").str.len_chars().mean().alias("panjang_karakter_mean"),
    pl.col("text").str.len_chars().max().alias("panjang_karakter_max"),
    pl.col("text").str.count_matches(r"\S+").mean().alias("jumlah_kata_mean"),
    (pl.col("text") == "").sum().alias("string_kosong"),
    (pl.col("text").str.strip_chars() == "").sum().alias("string_hanya_spasi"),
).collect()

print(f"Waktu komputasi: {time.time() - start:.2f} detik")
stats

Waktu komputasi: 55.93 detik


jumlah_baris,panjang_karakter_min,panjang_karakter_median,panjang_karakter_mean,panjang_karakter_max,jumlah_kata_mean,string_kosong,string_hanya_spasi
u32,u32,f64,f64,u32,f64,u32,u32
19483721,0,87.0,161.354069,180383,22.710027,1000001,1000001


## 7. Pemeriksaan missing/null values

`null` dihitung per kolom dengan agregasi lazy. Untuk kolom teks juga diperiksa jumlah string kosong,
karena teks dapat berupa string kosong tanpa bernilai `null`.

In [8]:
null_counts = lf.select(
    pl.len().alias("total_baris"),
    *[pl.col(name).is_null().sum().alias(name) for name in schema.names()],
).collect()

null_check = null_counts.unpivot(
    index="total_baris", variable_name="kolom", value_name="jumlah_null"
).with_columns(
    (pl.col("jumlah_null") / pl.col("total_baris") * 100).round(4).alias("persen_null")
)

null_check

total_baris,kolom,jumlah_null,persen_null
u32,str,u32,f64
19483721,"""text""",0,0.0


## 8. Jumlah data per file

Setiap file Parquet dibaca secara terpisah dengan `scan_parquet()` hanya untuk menghitung jumlah baris
dan ukuran file.

In [9]:
start = time.time()
per_file_rows = []
for path in files:
    jumlah = pl.scan_parquet(path).select(pl.len()).collect().item()
    per_file_rows.append(
        {
            "file": os.path.basename(path),
            "ukuran_byte": os.path.getsize(path),
            "jumlah_baris": jumlah,
        }
    )

per_file = pl.DataFrame(per_file_rows).with_columns(
    (pl.col("ukuran_byte") / 1_000_000).round(2).alias("ukuran_MB"),
    (
        pl.col("jumlah_baris") / pl.col("jumlah_baris").sum() * 100
    )
    .round(2)
    .alias("persen_baris"),
)

total_per_file = per_file["jumlah_baris"].sum()

print(f"Waktu komputasi: {time.time() - start:.2f} detik")
print(f"Total baris dari semua file : {total_per_file:,}")
print(f"Sama dengan jumlah baris data: {total_per_file == jumlah_baris}")

per_file

Waktu komputasi: 0.33 detik
Total baris dari semua file : 19,483,721
Sama dengan jumlah baris data: True


file,ukuran_byte,jumlah_baris,ukuran_MB,persen_baris
str,i64,i64,f64,f64
"""train-00000-of-00007.parquet""",272361891,2783389,272.36,14.29
"""train-00001-of-00007.parquet""",268977483,2783389,268.98,14.29
"""train-00002-of-00007.parquet""",271235387,2783389,271.24,14.29
"""train-00003-of-00007.parquet""",270602383,2783389,270.6,14.29
"""train-00004-of-00007.parquet""",272342018,2783389,272.34,14.29
"""train-00005-of-00007.parquet""",272175219,2783388,272.18,14.29
"""train-00006-of-00007.parquet""",272668591,2783388,272.67,14.29


## 9. Ringkasan hasil profiling

In [10]:
statistik = stats.to_dicts()[0]

ringkasan = pl.DataFrame(
    {
        "metrik": [
            "Path dataset",
            "Jumlah file Parquet",
            "Ukuran total (byte)",
            "Ukuran total (MB)",
            "Jumlah baris",
            "Jumlah kolom",
            "Kolom dan tipe data",
            "Jumlah nilai null",
            "Persen nilai null",
            "String kosong",
            "String hanya spasi",
            "Panjang teks min (karakter)",
            "Panjang teks median (karakter)",
            "Panjang teks mean (karakter)",
            "Panjang teks maks (karakter)",
            "Rata-rata jumlah kata",
            "Baris per file (min - maks)",
        ],
        "nilai": [
            DATA_PATH,
            str(len(files)),
            f"{per_file['ukuran_byte'].sum():,}",
            f"{per_file['ukuran_byte'].sum() / 1_000_000:,.2f}",
            f"{jumlah_baris:,}",
            str(jumlah_kolom),
            ", ".join(
                f"{k} ({t})" for k, t in zip(tipe_data["kolom"], tipe_data["tipe_data"])
            ),
            f"{int(null_check['jumlah_null'].sum()):,}",
            f"{null_check['persen_null'].max():.4f}%",
            f"{int(statistik['string_kosong']):,}",
            f"{int(statistik['string_hanya_spasi']):,}",
            f"{int(statistik['panjang_karakter_min']):,}",
            f"{statistik['panjang_karakter_median']:,.0f}",
            f"{statistik['panjang_karakter_mean']:,.2f}",
            f"{int(statistik['panjang_karakter_max']):,}",
            f"{statistik['jumlah_kata_mean']:,.2f}",
            f"{per_file['jumlah_baris'].min():,} - {per_file['jumlah_baris'].max():,}",
        ],
    }
)

ringkasan

metrik,nilai
str,str
"""Path dataset""","""data/raw/data/*.parquet"""
"""Jumlah file Parquet""","""7"""
"""Ukuran total (byte)""","""1,900,362,972"""
"""Ukuran total (MB)""","""1,900.36"""
"""Jumlah baris""","""19,483,721"""
…,…
"""Panjang teks median (karakter)""","""87"""
"""Panjang teks mean (karakter)""","""161.35"""
"""Panjang teks maks (karakter)""","""180,383"""


## 10. Catatan hasil

- **Skala dataset memenuhi Milestone 1**: file total ± 1,9 GB (≥ 500 MB) dan
  19.483.721 baris (> 1.000.000 baris).
- **Schema**: satu kolom `text` bertipe `String`; tidak ada kolom waktu/geografis,
  sehingga analisis temporal/spasial tidak relevan untuk dataset ini.
- **Null**: tidak ditemukan nilai `null` pada kolom `text` (0%). Namun terdapat string kosong,
  yang jumlahnya tercatat pada ringkasan di atas; hal ini merupakan temuan profiling
  yang akan ditangani pada tahap cleaning (Milestone 2), bukan pada Milestone 1.
- **Distribusi panjang teks**: panjang teks bervariasi (min sampai maks tercatat pada ringkasan);
  rata-rata teks jauh lebih besar dari median, menunjukkan sebagian teks sangat panjang
  dibanding teks tipikal.
- **Distribusi per file**: ketujuh file berukuran dan berisi baris yang hampir merata
  (± 2,78 juta baris per file).
- **Efisiensi memori**: notebook hanya mengumpulkan hasil agregat, schema, dan 5 baris preview;
  seluruh 19.483.721 baris tidak pernah dimuat sekaligus ke memori.
- Data raw tidak diubah; notebook ini bersifat baca-saja (`read-only`) terhadap `data/raw/`.